# 気象条件なども追加していきたい！

In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore", category=UserWarning)

In [19]:
# trainデータの読み込み
df_train = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/train.csv", encoding="cp932")
# 糸魚川駅、富山駅での通常車両の新幹線の車両への着雪量データであるフラグを付与
df_train["回送列車フラグ"] = 0

In [20]:
df_train

,年月日,列車番号,停車駅名,フェンダー部分(東京方向),台車部分,フェンダー部分(金沢方向),合計,回送列車フラグ
0,2016-01-19,3500E,富山,0.0,0.0,0.000000,0.000000,0
1,2016-01-19,562E,富山,0.0,0.0,0.000000,0.000000,0
2,2016-01-19,560E,糸魚川,0.0,0.0,0.000000,0.000000,0
3,2016-01-19,560E,富山,0.0,0.0,0.002986,0.002986,0
4,2016-01-19,558E,糸魚川,0.0,0.0,0.000000,0.000000,0
...,...,...,...,...,...,...,...,...
15310,2016-12-31,554E,糸魚川,0.0,0.0,0.000000,0.000000,0
15311,2016-12-31,574E,糸魚川,0.0,0.0,0.000000,0.000000,0
15312,2016-12-31,576E,富山,0.0,0.0,0.000000,0.000000,0
15313,2016-12-31,558E,富山,0.0,0.0,0.000000,0.000000,0


In [21]:
# 回送列車データの読み込み
df_through = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/out_of_service.csv", encoding="cp932")
# 糸魚川駅、富山駅での回送車両への着雪量データであるフラグを付与
df_through["回送列車フラグ"] = 1

In [22]:
# 通常列車、回送列車のデータを結合
df_all = pd.concat([df_train, df_through], ignore_index=True)

In [23]:
# ダイヤ情報の読み込み
df_dia = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/diagram.csv", encoding="cp932")
# df_dia

In [24]:
df_dia_T = df_dia.T
df_dia_T = df_dia_T.replace('↓', '通過')

df_dia_T.columns = df_dia_T.iloc[0]
df_dia_T = df_dia_T.iloc[1:]

# df_dia_T

In [25]:
# 列車情報とダイア情報を結合
df_all = pd.merge(df_all, df_dia_T, how='left', left_on='列車番号', right_index=True)
# df_all

In [26]:
# 金沢着雪ゼロ列車
df_zero = pd.read_csv("/home/keiseki/JR_train_snow/20.Data/kanazawa_nosnow.csv", encoding="cp932", header=None, names=["列車番号"])

# df_zeroに記載されている列車番号に「金沢着雪ゼロ列車」フラグを付与
df_zero["金沢着雪ゼロ列車フラグ"] = 1
# df_zero

In [27]:
df_merge = pd.merge(df_all, df_zero, how='left', on='列車番号')
df_merge["金沢着雪ゼロ列車フラグ"] = df_merge["金沢着雪ゼロ列車フラグ"].fillna(0)
df_merge

,年月日,列車番号,停車駅名,フェンダー部分(東京方向),台車部分,フェンダー部分(金沢方向),合計,回送列車フラグ,停車時刻,金沢,新高岡,富山,黒部宇奈月温泉,糸魚川,上越妙高,飯山,長野,金沢着雪ゼロ列車フラグ
0,2016-01-19,3500E,富山,0.000000,0.000000,0.000000,0.000000,0,NaN,6:00,通過,6:19,通過,通過,通過,通過,7:07,0.0
1,2016-01-19,562E,富山,0.000000,0.000000,0.000000,0.000000,0,NaN,11:56,12:10,12:19,12:32,12:46,12:59,通過,13:20,0.0
2,2016-01-19,560E,糸魚川,0.000000,0.000000,0.000000,0.000000,0,NaN,10:56,11:10,11:19,11:32,11:46,11:59,12:11,12:24,1.0
3,2016-01-19,560E,富山,0.000000,0.000000,0.002986,0.002986,0,NaN,10:56,11:10,11:19,11:32,11:46,11:59,12:11,12:24,1.0
4,2016-01-19,558E,糸魚川,0.000000,0.000000,0.000000,0.000000,0,NaN,9:21,9:35,9:45,9:57,10:11,10:25,10:37,11:00,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15334,2016-01-25,NaN,糸魚川,0.000048,0.002732,0.003908,0.006688,1,09:25:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15335,2016-01-25,NaN,糸魚川,0.000221,0.004932,0.005369,0.010521,1,08:30:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15336,2016-01-25,NaN,糸魚川,0.000136,0.005393,0.002608,0.008136,1,08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
15337,2016-01-25,NaN,糸魚川,0.002262,0.017006,0.000122,0.019390,1,07:42:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0


In [28]:
df_merge.to_pickle("/home/keiseki/JR_train_snow/20.Data/train_data_all.pkl")

---
testデータにも同じ特徴量を極力付与

In [29]:
test_path = '../20.Data/test.csv'
df_test = pd.read_csv(test_path, encoding='cp932', index_col=0)

In [30]:
# 列車情報とダイア情報を結合
df_test_all = pd.merge(df_test, df_dia_T, how='left', left_on='列車番号', right_index=True)
# 金沢着雪ゼロ列車のフラグを付与
df_test_merge = pd.merge(df_test_all, df_zero, how='left', on='列車番号')
df_test_merge["金沢着雪ゼロ列車フラグ"] = df_test_merge["金沢着雪ゼロ列車フラグ"].fillna(0)

# pickle出力
df_test_merge.to_pickle("/home/keiseki/JR_train_snow/20.Data/test_data_all(0910).pkl")

---
気象データの読み込みと日付単位への分割

In [31]:
# 読み込み
df_weather = pd.read_csv(
    "/home/keiseki/JR_train_snow/20.Data/weather.csv",
    encoding="cp932",
    index_col=0
)
df_weather = df_weather.reset_index()

# 年月日時をdatetime型にする
df_weather["年月日時"] = pd.to_datetime(df_weather["年月日時"])

# 日付と時刻を分離
df_weather["日付"] = df_weather["年月日時"].dt.date
df_weather["時刻"] = df_weather["年月日時"].dt.strftime("%-H:%M")

# 横持ちする気象項目
value_cols = [
    col for col in df_weather.columns
    if col not in ["年月日時", "日付", "時刻", "地点"]
]

# 日付 × 地点 × 時刻 を横持ち
df_weather_wide = df_weather.pivot(
    index="日付",
    columns=["地点", "時刻"],
    values=value_cols
)

# MultiIndexになったカラムを
# 「地点_項目_時刻」に変換
df_weather_wide.columns = [
    f"{place}_{item}_{time}"
    for item, place, time in df_weather_wide.columns
]

# indexを通常の列に戻す
df_weather_wide = df_weather_wide.reset_index()
# 日付単位の気象データを出力
df_weather_wide.to_pickle("/home/keiseki/JR_train_snow/20.Data/weather_data_daily.pkl")

In [32]:
# 日付単位の気象データをtrain/testデータに結合
df_train_all = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/train_data_all.pkl")
df_test_all = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/test_data_all(0910).pkl")
df_weather = pd.read_pickle("/home/keiseki/JR_train_snow/20.Data/weather_data_daily.pkl")

# 年月日をdatetime型に変換
df_train_all["年月日"] = pd.to_datetime(df_train_all["年月日"])
df_test_all["年月日"] = pd.to_datetime(df_test_all["年月日"])
df_weather["日付"] = pd.to_datetime(df_weather["日付"])


df_train_all = pd.merge(df_train_all, df_weather, how='left', left_on='年月日', right_on='日付', indicator=True)
df_test_all = pd.merge(df_test_all, df_weather, how='left', left_on='年月日', right_on='日付', indicator=True)

In [33]:
df_train_all.columns = (
    df_train_all.columns
    .str.replace(":", "_", regex=False)
    .str.replace("(", "_", regex=False)
    .str.replace(")", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("㎡", "m2", regex=False)
)
df_test_all.columns = (
    df_test_all.columns
    .str.replace(":", "_", regex=False)
    .str.replace("(", "_", regex=False)
    .str.replace(")", "_", regex=False)
    .str.replace("/", "_", regex=False)
    .str.replace("㎡", "m2", regex=False)
)

In [34]:
df_train_all.to_pickle("/home/keiseki/JR_train_snow/20.Data/train_data_all_with_weather.pkl")
df_test_all.to_pickle("/home/keiseki/JR_train_snow/20.Data/test_data_all_with_weather.pkl")